In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
### 所需要的所有文件：
# 物流的发货清单
# 财务的发货清单
# 核算价
# PLM的生命周期全表
### 输出的所有文件
# 统计周期内产品核算价汇总 用于物料精简报告，因为里面有国内国外
# 合并物流-财务-产品组-核算价-国内-用于低效-长尾  ，用于低效-长尾报告，里面只有国内
# 单型号贡献-统计值



### MAP关系汇总1、渠道对照 2、最终表格产品类别对应的产品组集合 3、产品组集合

In [2]:
# 物流的渠道对照关系清洗用
month = 202511
Channel_map = {
'工程': '工程',
'零售':'零售',
'电商不可售':'电商',
'电商':'电商',
'内部处理通用':'电商',
'借出渠道':'非零售工程电商',
'新品':'电商',
'战略电商':'电商',
'转出渠道':'非零售工程电商',
'每誉':'每誉',
'渠道':'非零售工程电商',
'海外':'海外',
'调出渠道':'非零售工程电商',
'非零售工程电商':'非零售工程电商',
'非零售工程电商':'非零售工程电商'
}
#用于合并计算
productgroupset_map = {
    '吸油烟机':['吸油烟机'],
    '灶具':['灶具'],
    '烤箱':['烤箱'],
    '蒸箱':['蒸箱'],
    '微波炉':['微波炉'],
    '蒸烤烹饪机':['蒸烤烹饪机'],
    '蒸烤微烹饪机':['蒸烤微烹饪机'],
    '蒸微':['蒸微'],
    '蒸烤微合计':['烤箱','蒸箱','微波炉','蒸烤烹饪机','蒸烤微烹饪机','蒸微'],
    '灶消烹饪机':['灶消烹饪机'],
    '灶蒸烹饪机':['灶蒸烹饪机'],
    '灶蒸烤烹饪机':['灶蒸烤烹饪机'],
    '灶烤烹饪机':['灶烤烹饪机'],
    '灶集成':['灶消烹饪机','灶蒸烹饪机','灶蒸烤烹饪机','灶烤烹饪机'],
    '烹饪产品线合计':['灶具','烤箱','蒸箱','微波炉','蒸烤烹饪机','蒸烤微烹饪机','蒸微','灶消烹饪机','灶蒸烹饪机','灶蒸烤烹饪机','灶烤烹饪机'],
    '消毒柜':['消毒柜'],
    '热水器':['热水器'],
    '两用炉':['两用炉'],
    '热水器两用炉合计':['热水器','两用炉'],
    '家用净水机':['家用净水机'],
    '商用净水机':['商用净水机'],
    '净热产品线合计':['热水器','两用炉','家用净水机','商用净水机'],
    '水槽洗碗机':['水槽洗碗机'],
    '嵌入式洗碗机':['嵌入式洗碗机'],
    '洗碗机产品线合计':['水槽洗碗机','嵌入式洗碗机'],
    '国内合计':['吸油烟机','灶具','烤箱','蒸箱','微波炉','蒸烤烹饪机','蒸烤微烹饪机','蒸微','灶消烹饪机','灶蒸烹饪机','灶蒸烤烹饪机','灶烤烹饪机','消毒柜','热水器','两用炉','家用净水机','商用净水机','水槽洗碗机','嵌入式洗碗机'],
}
# 统计的产品组
productgroup_list = ['吸油烟机','灶具','烤箱','蒸箱','微波炉','蒸烤烹饪机','蒸烤微烹饪机','蒸微','灶消烹饪机','灶蒸烹饪机','灶蒸烤烹饪机','灶烤烹饪机','消毒柜','热水器','两用炉','热水器两用炉合计','家用净水机','商用净水机','净热产品线合计','水槽洗碗机','嵌入式洗碗机','洗碗机产品线合计','国内合计']


### 获取物流体系的文件夹中的文件路劲和对应规则的sheetname。并将这些数据进行上写拼接（只保留了编码，渠道，发货数量）

In [3]:
#将这些excel的表格数据进行上下拼接，只要字段：商品编码、商品名称、渠道、实际出库数量
folder_path = fr"D:\000物料报表\{month}\单型号贡献-低效-长尾\物流发货"  # 请替换为实际的文件夹路径
excel_files = []
for root, dirs, files in os.walk(folder_path):
    for file in files:
        sheet_names = file[5:].replace(file[-5:],'')
        if (file.endswith('.xlsx') or file.endswith('.xls')) and not file.startswith('~$'):
            file_path = os.path.join(root, file)
            excel_files.append((file, sheet_names, file_path))
print(excel_files)

[('2025年10月明细.xlsx', '10月明细', 'D:\\000物料报表\\202511\\单型号贡献-低效-长尾\\物流发货\\2025年10月明细.xlsx'), ('2025年1月明细.xlsx', '1月明细', 'D:\\000物料报表\\202511\\单型号贡献-低效-长尾\\物流发货\\2025年1月明细.xlsx'), ('2025年2月明细.xlsx', '2月明细', 'D:\\000物料报表\\202511\\单型号贡献-低效-长尾\\物流发货\\2025年2月明细.xlsx'), ('2025年3月明细.xlsx', '3月明细', 'D:\\000物料报表\\202511\\单型号贡献-低效-长尾\\物流发货\\2025年3月明细.xlsx'), ('2025年4月明细.xlsx', '4月明细', 'D:\\000物料报表\\202511\\单型号贡献-低效-长尾\\物流发货\\2025年4月明细.xlsx'), ('2025年5月明细.xlsx', '5月明细', 'D:\\000物料报表\\202511\\单型号贡献-低效-长尾\\物流发货\\2025年5月明细.xlsx'), ('2025年6月明细.xlsx', '6月明细', 'D:\\000物料报表\\202511\\单型号贡献-低效-长尾\\物流发货\\2025年6月明细.xlsx'), ('2025年7月明细.xlsx', '7月明细', 'D:\\000物料报表\\202511\\单型号贡献-低效-长尾\\物流发货\\2025年7月明细.xlsx'), ('2025年8月明细.xlsx', '8月明细', 'D:\\000物料报表\\202511\\单型号贡献-低效-长尾\\物流发货\\2025年8月明细.xlsx'), ('2025年9月明细.xlsx', '9月明细', 'D:\\000物料报表\\202511\\单型号贡献-低效-长尾\\物流发货\\2025年9月明细.xlsx')]


In [4]:
#如果有报错请提示报错信息
df = pd.DataFrame()
for file in excel_files:
    try:
        df_temp = pd.read_excel(file[2], sheet_name=file[1])
        print(f'成功读取{file[0]}的{file[1]}表')
    except:
        print(f'读取{file[0]}的{file[1]}表失败')
    if '实际总数量' in df_temp.columns:
        df_temp = df_temp.rename(columns={'实际总数量':'实际出库数量'})
    df_temp = df_temp[['商品编码', '渠道', '实际出库数量']]
    df = pd.concat([df, df_temp], axis=0).reset_index(drop=True)
df = df.dropna(how='all').reset_index(drop=True)  # 仅当一行所有值都是NaN时才删除
df['商品编码'] = df['商品编码'].astype(str)
df['渠道'].value_counts()


成功读取2025年10月明细.xlsx的10月明细表
成功读取2025年1月明细.xlsx的1月明细表
成功读取2025年2月明细.xlsx的2月明细表
成功读取2025年3月明细.xlsx的3月明细表
成功读取2025年4月明细.xlsx的4月明细表
成功读取2025年5月明细.xlsx的5月明细表
成功读取2025年6月明细.xlsx的6月明细表
成功读取2025年7月明细.xlsx的7月明细表
成功读取2025年8月明细.xlsx的8月明细表
成功读取2025年9月明细.xlsx的9月明细表


渠道
零售        390588
工程         16940
电商          7855
电商不可售       6616
海外          1771
每誉           152
战略电商         129
新品            67
内部处理通用        15
借出渠道          14
商净             7
渠道             4
调出渠道           2
转出渠道           2
米博新零售          1
Name: count, dtype: int64

### 处理历史残留（财务的发货数据，将其转化为编码，名称，渠道，出库数量的形式）

In [5]:
#将这些excel的表格数据进行上下拼接，只要字段：商品编码、商品名称、渠道、实际出库数量
folder_path = fr"D:\000物料报表\{month}\单型号贡献-低效-长尾\财务发货"  # 请替换为实际的文件夹路径
excel_files = []
for root, dirs, files in os.walk(folder_path):
    for file in files:
        sheet_names = file[5:].replace(file[-5:],'')
        if (file.endswith('.xlsx') or file.endswith('.xls')) and not file.startswith('~$'):
            file_path = os.path.join(root, file)
            excel_files.append((file, sheet_names, file_path))
print(excel_files)

[('2024年11月明细.xlsx', '11月明细', 'D:\\000物料报表\\202511\\单型号贡献-低效-长尾\\财务发货\\2024年11月明细.xlsx'), ('2024年12月明细.xlsx', '12月明细', 'D:\\000物料报表\\202511\\单型号贡献-低效-长尾\\财务发货\\2024年12月明细.xlsx')]


In [6]:
caiwu_shouru = {'商品编码':[],'渠道':[],'实际出库数量':[]}
for file in excel_files:
    try:
        df_temp = pd.read_excel(file[2], sheet_name=file[1])
        print(f'成功读取{file[0]}的{file[1]}表')
    except:
        print(file[0], file[1])
    for index,row in df_temp.iterrows():
        if row['零售'] > 0:
            caiwu_shouru['商品编码'].append(row['物料编码'])
            caiwu_shouru['渠道'].append('零售')
            caiwu_shouru['实际出库数量'].append(row['零售'])
        if row['工程'] >0:
            caiwu_shouru['商品编码'].append(row['物料编码'])
            caiwu_shouru['渠道'].append('工程')
            caiwu_shouru['实际出库数量'].append(row['工程'])
        if row['电商'] >0:
            caiwu_shouru['商品编码'].append(row['物料编码'])
            caiwu_shouru['渠道'].append('电商')
            caiwu_shouru['实际出库数量'].append(row['电商'])
        if row['合计-发货'] - row['零售'] - row['工程'] - row['电商'] != 0:
            caiwu_shouru['商品编码'].append(row['物料编码'])
            caiwu_shouru['渠道'].append('非零售工程电商')
            caiwu_shouru['实际出库数量'].append(row['合计-发货'] - row['零售'] - row['工程'] - row['电商'])

df_caiwu = pd.DataFrame(caiwu_shouru)
df_caiwu['商品编码'] = df_caiwu['商品编码'].astype(str)
df_caiwu['渠道'].value_counts()


成功读取2024年11月明细.xlsx的11月明细表
成功读取2024年12月明细.xlsx的12月明细表


渠道
零售         1029
电商          797
工程          476
非零售工程电商      16
Name: count, dtype: int64

### 将财务和物流的数据进行拼接,并进行渠道清洗

In [12]:
df0 = pd.concat([df, df_caiwu], axis=0).reset_index(drop=True)
df0.to_excel(r'C:\Users\zhangbon\Desktop\总发货.xlsx', index=False)
df0['物料编码'] = df0['商品编码'].astype(str)
df0['渠道'] = df0['渠道'].map(Channel_map).fillna('非零售工程电商')
df0 = df0[(df0['渠道']!='无') & 
        (df0['物料编码'].str.len()>=10)
        ].reset_index(drop=True)
df0['渠道'].value_counts()
# df0


渠道
零售         391617
工程          17416
电商          15479
海外           1771
每誉            152
非零售工程电商        24
Name: count, dtype: int64

In [13]:
df0.to_excel(r'C:\Users\zhangbon\Desktop\发货数据.xlsx', index=False)

### 将产品组信息、核算价匹配进去

In [14]:
# 复制df
df1 = df0.copy()
df1['物料编码'] = df1['物料编码'].astype(str)
df1['实际出库数量'] = df1['实际出库数量'].astype(float)

# PLM产品数据导入
df_product_group = pd.read_excel(fr"D:\000物料报表\{month}\单型号贡献-低效-长尾\产品生命周期状态全表.xlsx")
df_product_group[['物料号','标准型号','国内/海外']] = df_product_group[['物料号','标准型号','国内/海外']].astype(str)
df_product_group = df_product_group[df_product_group['物料号'].str.len()>10]
df_product_group['物料号'] = df_product_group['物料号'].apply(lambda x: x[:13])
# df_product_group



e:\python\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [19]:
#产品组对照关系
product_group_map = dict(zip(df_product_group['物料号'],df_product_group['产品组']))
#标准型号对照关系
product_standard = dict(zip(df_product_group['物料号'],df_product_group['标准型号']))
#记录国内/海外状态
product_country = dict(zip(df_product_group['物料号'],df_product_group['国内/海外']))
#记录生命周期状态
product_status = dict(zip(df_product_group['物料号'],df_product_group['产品状态']))


In [20]:
# 导入财务的核算价
df_price = pd.read_excel(fr"D:\000物料报表\{month}\单型号贡献-低效-长尾\核算价.xlsx")
df_price['产品编码'] = df_price['产品编码'].astype(str)
df_price['系统核算价'] = df_price['系统核算价'].astype(float)
# df_price
product_price_map = dict(zip(df_price['产品编码'],df_price['系统核算价']))

In [21]:

df1['产品组'] = df1['商品编码'].map(product_group_map)
df1['系统核算价'] = df1['商品编码'].map(product_price_map)
df1['核算价'] = df1['系统核算价'] * df1['实际出库数量']
df1['标准型号'] = df1['商品编码'].map(product_standard)
df1['国内/海外'] = df1['商品编码'].map(product_country)
df1['生命周期状态'] = df1['商品编码'].map(product_status)
df1 = df1[df1['商品编码'].str.startswith('10')].reset_index(drop=True)
#这里要提示哪些数据的系统核算价是空的
print(f'国内没有系统核算价的是这些数据\n{df1[(df1['系统核算价'].isnull()&(df1['国内/海外']=='国内'))]['商品编码'].drop_duplicates()}')
print(len(df1))
df1

国内没有系统核算价的是这些数据
Series([], Name: 商品编码, dtype: object)
421630


,商品编码,渠道,实际出库数量,物料编码,产品组,系统核算价,核算价,标准型号,国内/海外,生命周期状态
0,1001001500116,零售,12.0,1001001500116,吸油烟机,3358.0,40296.0,Z8T,国内,退市预警
1,1009000600033,零售,1.0,1009000600033,蒸烤烹饪机,3450.0,3450.0,ZK50-02-F1,国内,量产
2,1009000500035,零售,3.0,1009000500035,灶蒸烤烹饪机,5280.0,15840.0,JZT-ZK46-X2,国内,量产
3,1001001500131,零售,6.0,1001001500131,吸油烟机,2988.0,17928.0,02-Z6TA,国内,退市预警
4,1002003700049,零售,5.0,1002003700049,灶具,2550.0,12750.0,H8B,国内,量产
...,...,...,...,...,...,...,...,...,...,...
421625,1021000400003,非零售工程电商,1866.0,1021000400003,地面手持式清洁机,4200.0,7837200.0,nan,国内,量产
421626,1021000400006,非零售工程电商,1644.0,1021000400006,地面手持式清洁机,3346.0,5500824.0,QX-V7Plus,国内,小批量
421627,1021000400007,非零售工程电商,180.0,1021000400007,地面手持式清洁机,3522.0,633960.0,nan,国内,小批量
421628,1021000400008,非零售工程电商,2.0,1021000400008,地面手持式清洁机,3522.0,7044.0,QX-V8,国内,量产


### 这里导出的数据，只有整机有产品组的值，其他产品组为空,这个用于物料精简（零件分析）报告的分析，用于计算收入

In [22]:
# 物流和财务的发货收入中是包含了1012开头的样机的，PLM中是只有成品的，所以这里样机的产品组是空的，样机不在我们报告的统计范围内
# 这份导出的数据中是包含了国内国外的整机以及样机等数据，（样机等不会有产品组信息，因为PLM产品生命周期表中是没有样机的）
df1.to_excel(fr'D:\000物料报表\{month}\单物料产值\统计周期内产品核算价汇总.xlsx',index=False)


### 保留范围内的产品组(这样是会只剩下整机)、国内产品、停止发货以前（不含停止发货）、三大渠道，这里要注意补充核算价！！！！！（在这里会输出一个文件用于后续低效和长尾的分析）

In [ ]:
out_df = df1.copy()
out_df = out_df[(out_df['国内/海外']=='国内') & 
                (out_df['生命周期状态'].isin(['开发','停止生产','量产','停止销售','小批量','样机','退市预警','创建',])) 
                ]
out_df = out_df[out_df['产品组'].isin(productgroup_list)]
#这里要提示哪些数据的系统核算价是空的
print(f'没有系统核算价的是这些数据\n{out_df[out_df['系统核算价'].isnull()]['商品编码'].drop_duplicates()}')
print(len(out_df))

# 这里导出数据用于低效和长尾的分析。因为低效和长尾只看国内的成品情况（不包含样机，且在我们的统计产品组范围内）
out_df.to_excel(fr'D:\000物料报表\{month}\单型号贡献-低效-长尾\合并物流-财务-产品组-核算价-国内-用于低效-长尾.xlsx',index=False)
out_df


SyntaxError: invalid syntax (663192763.py, line 4)

### 费超洁，系列的单型号产值贡献

In [16]:
df_series = pd.read_excel(r'C:\Users\zhangbon\Desktop\202509年数据统计（含冰箱）.xlsx',sheet_name='型号分类明显')
df_series['物料编码'] = df_series['物料编码'].astype(str).str[:13]
df_series


,产品组描述,系列1,物料编码,物料名称
0,热水器,电商恒温13L,1004002100008,JSQ25-02-P13T1-FR-12T
1,热水器,电商恒温16L,1004001300028,JSQ30-02-MS16T1-FR-12T
2,热水器,电商恒温16L,1004001300029,JSQ30-02-P16T3-FR
3,热水器,电商恒温13L,1004002100009,JSQ25-02-P13D1-FR-12T
4,热水器,家装D系列,1004001300014,JSQ30-02-D1663-12T
...,...,...,...,...
707,集成式洗碗机,工程专供,1124000200150,嵌洗模块_JBID15E-X1A
708,集成式洗碗机,工程专供,1024000200024,JPID6E-03-M-Y3
709,集成式洗碗机,工程专供,1024000200025,JPID6E-03-M-Y3L
710,集成式洗碗机,智慧水槽,1024000200018,JBS2T-X20Max


In [39]:
df_series_channel = pd.merge(df_series,pd.DataFrame({'渠道':['零售','工程','电商']}),how='cross')
df_series_channel

,产品组描述,系列1,物料编码,物料名称,渠道
0,热水器,电商恒温13L,1004002100008,JSQ25-02-P13T1-FR-12T,零售
1,热水器,电商恒温13L,1004002100008,JSQ25-02-P13T1-FR-12T,工程
2,热水器,电商恒温13L,1004002100008,JSQ25-02-P13T1-FR-12T,电商
3,热水器,电商恒温16L,1004001300028,JSQ30-02-MS16T1-FR-12T,零售
4,热水器,电商恒温16L,1004001300028,JSQ30-02-MS16T1-FR-12T,工程
...,...,...,...,...,...
2131,集成式洗碗机,智慧水槽,1024000200018,JBS2T-X20Max,工程
2132,集成式洗碗机,智慧水槽,1024000200018,JBS2T-X20Max,电商
2133,集成式洗碗机,智慧水槽,1024000200019,JBS2T-X20MaxL,零售
2134,集成式洗碗机,智慧水槽,1024000200019,JBS2T-X20MaxL,工程


In [49]:
for index, row in df_series_channel.iterrows():
    df_series_channel.loc[index,'核算价总计'] = out_df[(out_df['商品编码'] == row['物料编码'])&(out_df['渠道'] == row['渠道'])]['核算价'].sum()
df_series_channel
df_series_channel.to_excel(fr'C:\Users\zhangbon\Desktop\系列渠道维度单型号贡献-1.xlsx',index=False)

In [52]:
df_series_out1 = df_series_channel[['产品组描述','系列1','渠道']].drop_duplicates().reset_index(drop=True)
df_series_out1 
for index, row in df_series_out1.iterrows():
    df_series_out1.loc[index,'核算价总计'] = df_series_channel[(df_series_channel['产品组描述'] == row['产品组描述'])&(df_series_channel['系列1'] == row['系列1'])&(df_series_channel['渠道'] == row['渠道'])&(df_series_channel['核算价总计'] > 0)]['核算价总计'].sum()/10000
    df_series_out1.loc[index,'产品型号计数'] = df_series_channel[(df_series_channel['产品组描述'] == row['产品组描述'])&(df_series_channel['系列1'] == row['系列1'])&(df_series_channel['渠道'] == row['渠道'])&(df_series_channel['核算价总计'] > 0)]['物料编码'].nunique()
df_series_out1
df_series_out1['各渠道单型号贡献'] = df_series_out1['核算价总计']/df_series_out1['产品型号计数']
df_series_out1

,产品组描述,系列1,渠道,核算价总计,产品型号计数,各渠道单型号贡献
0,热水器,电商恒温13L,零售,0.000,0.0,NaN
1,热水器,电商恒温13L,工程,0.000,0.0,NaN
2,热水器,电商恒温13L,电商,1037.235,3.0,345.745
3,热水器,电商恒温16L,零售,0.000,0.0,NaN
4,热水器,电商恒温16L,工程,0.000,0.0,NaN
...,...,...,...,...,...,...
181,集成式洗碗机,工程专供,工程,0.000,0.0,NaN
182,集成式洗碗机,工程专供,电商,0.000,0.0,NaN
183,集成式洗碗机,智慧水槽,零售,0.000,0.0,NaN
184,集成式洗碗机,智慧水槽,工程,0.000,0.0,NaN


In [53]:
df_series_out1.to_excel(fr'C:\Users\zhangbon\Desktop\系列渠道维度单型号贡献.xlsx',index=False)

In [ ]:
# df_series_out1 = df_series[['产品组描述','系列1']].drop_duplicates().reset_index(drop=True)
# df_series_out1 
# df_series_out2 = pd.merge(df_series_out1,pd.DataFrame({'渠道':['零售','工程','电商']}),how='cross')
# df_series_out2
# # for index, row in df_series_out1.iterrows():
# #     df_series_out1.loc[index,'核算价总计'] = df_series[(df_series['产品组描述'] == row['产品组描述'])&(df_series['系列1'] == row['系列1'])]['核算价总计'].sum()
# for index ,row in df_series_out2.iterrows():
#     df_series_out2.loc[index,'核算价总计'] = df_series[(df_series['产品组描述'] == row['产品组描述'])&(df_series['系列1'] == row['系列1'])&(df_series['渠道'] == row['渠道'])]['核算价总计'].sum()


,产品组描述,系列1,渠道
0,热水器,电商恒温13L,零售
1,热水器,电商恒温13L,工程
2,热水器,电商恒温13L,电商
3,热水器,电商恒温16L,零售
4,热水器,电商恒温16L,工程
...,...,...,...
181,集成式洗碗机,工程专供,工程
182,集成式洗碗机,工程专供,电商
183,集成式洗碗机,智慧水槽,零售
184,集成式洗碗机,智慧水槽,工程


### 构建最终的统计结果表

In [17]:
df_calu = pd.DataFrame()
df_calu['产品类别'] = productgroupset_map.keys()
for k,v in productgroupset_map.items():
    # 零售渠道的统计
    vals = out_df[(out_df['产品组'].isin(v))&(out_df['渠道'] == '零售')]['标准型号'].nunique()
    df_calu.loc[df_calu['产品类别']==k,'零售渠道标准型号数'] = vals
    vals = out_df[(out_df['产品组'].isin(v))&(out_df['渠道'] == '零售')]['核算价'].sum()
    df_calu.loc[df_calu['产品类别']==k,'零售渠道核算价'] = vals

    # 工程渠道的统计
    vals = out_df[(out_df['产品组'].isin(v))&(out_df['渠道'] == '工程')]['标准型号'].nunique()
    df_calu.loc[df_calu['产品类别']==k,'工程渠道标准型号数'] = vals
    vals = out_df[(out_df['产品组'].isin(v))&(out_df['渠道'] == '工程')]['核算价'].sum()
    df_calu.loc[df_calu['产品类别']==k,'工程渠道核算价'] = vals

    # 电商渠道的统计
    vals = out_df[(out_df['产品组'].isin(v))&(out_df['渠道'] == '电商')]['标准型号'].nunique()
    df_calu.loc[df_calu['产品类别']==k,'电商渠道标准型号数'] = vals
    vals = out_df[(out_df['产品组'].isin(v))&(out_df['渠道'] == '电商')]['核算价'].sum()
    df_calu.loc[df_calu['产品类别']==k,'电商渠道核算价'] = vals

    # 合计
    vals = out_df[(out_df['产品组'].isin(v))]['标准型号'].nunique()
    df_calu.loc[df_calu['产品类别']==k,'汇总标准型号数'] = vals
    vals = out_df[(out_df['产品组'].isin(v))]['核算价'].sum()
    df_calu.loc[df_calu['产品类别']==k,'汇总核算价'] = vals
df_calu

,产品类别,零售渠道标准型号数,零售渠道核算价,工程渠道标准型号数,工程渠道核算价,电商渠道标准型号数,电商渠道核算价,汇总标准型号数,汇总核算价
0,吸油烟机,94.0,4.111059e+09,71.0,9.264835e+08,131.0,2.608933e+09,172.0,7.646475e+09
1,灶具,66.0,1.902560e+09,60.0,4.690062e+08,88.0,1.340452e+09,108.0,3.712018e+09
2,烤箱,8.0,5.979212e+07,7.0,3.592640e+06,13.0,1.312160e+07,14.0,7.650636e+07
3,蒸箱,7.0,7.138789e+07,4.0,3.233750e+06,14.0,1.640138e+07,15.0,9.102302e+07
4,微波炉,4.0,1.218356e+07,3.0,3.817990e+06,4.0,4.916470e+06,5.0,2.091802e+07
5,蒸烤烹饪机,26.0,6.402838e+08,20.0,6.113453e+07,43.0,2.125358e+08,48.0,9.140467e+08
6,蒸烤微烹饪机,4.0,8.908920e+06,4.0,1.576500e+06,6.0,5.804663e+07,8.0,6.853205e+07
7,蒸微,0.0,0.000000e+00,0.0,0.000000e+00,0.0,0.000000e+00,0.0,0.000000e+00
8,蒸烤微合计,49.0,7.925563e+08,38.0,7.335541e+07,80.0,3.050219e+08,90.0,1.171026e+09
9,灶消烹饪机,3.0,3.574376e+07,0.0,0.000000e+00,3.0,1.217570e+06,3.0,3.696133e+07


#### 输出结果

In [18]:

df_calu[['零售渠道核算价','电商渠道核算价','工程渠道核算价','汇总核算价']] = df_calu[['零售渠道核算价','电商渠道核算价','工程渠道核算价','汇总核算价']]/10000
df_calu['零售单型号贡献值'] = df_calu['零售渠道核算价']/df_calu['零售渠道标准型号数']
df_calu['工程单型号贡献值'] = df_calu['工程渠道核算价']/df_calu['工程渠道标准型号数']
df_calu['电商单型号贡献值'] = df_calu['电商渠道核算价']/df_calu['电商渠道标准型号数']
df_calu['汇总单型号贡献值'] = df_calu['汇总核算价']/df_calu['汇总标准型号数']
df_calu.to_excel(fr'D:\000物料报表\{month}\单型号贡献-低效-长尾\单型号贡献-统计值.xlsx',index=False)
df_calu


,产品类别,零售渠道标准型号数,零售渠道核算价,工程渠道标准型号数,工程渠道核算价,电商渠道标准型号数,电商渠道核算价,汇总标准型号数,汇总核算价,零售单型号贡献值,工程单型号贡献值,电商单型号贡献值,汇总单型号贡献值
0,吸油烟机,94.0,411105.8886,71.0,92648.3456,131.0,260893.2968,172.0,7.646475e+05,4373.466900,1304.906276,1991.551884,4445.625180
1,灶具,66.0,190255.9670,60.0,46900.6248,88.0,134045.1806,108.0,3.712018e+05,2882.666167,781.677080,1523.240689,3437.053448
2,烤箱,8.0,5979.2120,7.0,359.2640,13.0,1312.1600,14.0,7.650636e+03,747.401500,51.323429,100.935385,546.474000
3,蒸箱,7.0,7138.7890,4.0,323.3750,14.0,1640.1380,15.0,9.102302e+03,1019.827000,80.843750,117.152714,606.820133
4,微波炉,4.0,1218.3560,3.0,381.7990,4.0,491.6470,5.0,2.091802e+03,304.589000,127.266333,122.911750,418.360400
5,蒸烤烹饪机,26.0,64028.3806,20.0,6113.4533,43.0,21253.5843,48.0,9.140467e+04,2462.630023,305.672665,494.269402,1904.263921
6,蒸烤微烹饪机,4.0,890.8920,4.0,157.6500,6.0,5804.6630,8.0,6.853205e+03,222.723000,39.412500,967.443833,856.650625
7,蒸微,0.0,0.0000,0.0,0.0000,0.0,0.0000,0.0,0.000000e+00,NaN,NaN,NaN,NaN
8,蒸烤微合计,49.0,79255.6296,38.0,7335.5413,80.0,30502.1923,90.0,1.171026e+05,1617.461829,193.040561,381.277404,1301.140147
9,灶消烹饪机,3.0,3574.3760,0.0,0.0000,3.0,121.7570,3.0,3.696133e+03,1191.458667,NaN,40.585667,1232.044333
